In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Section.csv
/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv
/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv
/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Header.csv


In [2]:
import pandas as pd
import numpy as np
import gc

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"

print("Reading column names...")

columns = pd.read_csv(API_PATH, nrows=0).columns

print(f"Columns found: {len(columns)}")

# Create dtype dictionary
dtype_dict = {}

for col in columns:
    if col == "SHA256":
        dtype_dict[col] = "string"
    else:
        dtype_dict[col] = np.uint8

print("Loading API dataset...")

api_df = pd.read_csv(
    API_PATH,
    dtype=dtype_dict,
    low_memory=False,
    engine="c"
)

gc.collect()

print("\nSuccessfully Loaded!\n")

print(api_df.info(memory_usage="deep"))

print("\nShape:", api_df.shape)

print("\nMemory Usage (MB):",
      round(api_df.memory_usage(deep=True).sum()/1024**2,2))

Reading column names...
Columns found: 21920
Loading API dataset...

Successfully Loaded!

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29505 entries, 0 to 29504
Columns: 21920 entries, SHA256 to setupdigethwprofilefriendlynameexw
dtypes: string(1), uint8(21919)
memory usage: 619.9 MB
None

Shape: (29505, 21920)

Memory Usage (MB): 619.94


In [3]:
# ===========================
# API Frequency Analysis
# ===========================

import pandas as pd

api_columns = api_df.columns.drop(["SHA256", "Type"])

api_usage = api_df[api_columns].sum()

print("="*70)
print("API Usage Statistics")
print("="*70)

print(api_usage.describe())

print("\nAPIs appearing exactly once :", (api_usage == 1).sum())
print("APIs appearing <5 times     :", (api_usage < 5).sum())
print("APIs appearing <10 times    :", (api_usage < 10).sum())
print("APIs appearing <20 times    :", (api_usage < 20).sum())
print("APIs appearing <50 times    :", (api_usage < 50).sum())

print("\nTop 30 Most Common APIs")
print(api_usage.sort_values(ascending=False).head(30))

print("\nTop 30 Least Common APIs")
print(api_usage.sort_values().head(30))

API Usage Statistics
count    21918.000000
mean        93.521352
std        545.252938
min          1.000000
25%          1.000000
50%          2.000000
75%          5.000000
max      13458.000000
dtype: float64

APIs appearing exactly once : 10290
APIs appearing <5 times     : 16039
APIs appearing <10 times    : 17824
APIs appearing <20 times    : 18839
APIs appearing <50 times    : 19544

Top 30 Most Common APIs
getprocaddress                 13458
corexemain                     12373
exitprocess                    11389
getlasterror                   10751
loadlibrarya                   10634
getcurrentprocess              10546
sleep                          10187
writefile                       9884
multibytetowidechar             9713
widechartomultibyte             9708
getmodulehandlea                9502
getcurrentthreadid              9376
closehandle                     8918
unhandledexceptionfilter        8877
gettickcount                    8825
leavecriticalsection       

In [4]:
from sklearn.feature_selection import VarianceThreshold

# -----------------------------------
# Separate Features and Labels
# -----------------------------------

X = api_df.drop(columns=["SHA256", "Type"])
y = api_df["Type"]

print("Original Features :", X.shape[1])

# -----------------------------------
# Variance Threshold
# Remove almost constant features
# -----------------------------------

selector = VarianceThreshold(threshold=0.001)

selector.fit(X)

selected_columns = X.columns[selector.get_support()]

print("Remaining Features :", len(selected_columns))
print("Removed Features   :", X.shape[1] - len(selected_columns))

# Save selected feature names
variance_features = list(selected_columns)

print("\nFirst 20 Selected Features:")
print(variance_features[:20])

Original Features : 21918
Remaining Features : 2720
Removed Features   : 19198

First 20 Selected Features:
['getaclinformation', 'getace', 'getsecuritydescriptordacl', 'regqueryvalueexa', 'regopenkeyexa', 'getsecurityinfo', 'isvalidsid', 'regclosekey', 'getnamedsecurityinfow', 'convertstringsecuritydescriptortosecuritydescriptorw', 'regsetvalueexw', 'getsecuritydescriptorsacl', 'setsecuritydescriptorowner', 'initializesecuritydescriptor', 'regcreatekeyexw', 'mapgenericmask', 'regqueryvalueexw', 'regopenkeyexw', 'adjusttokenprivileges', 'lookupprivilegevaluea']


In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

# ----------------------------------------------------
# Use only variance-selected features
# ----------------------------------------------------

X_selected = X[variance_features]

print("Features entering MI:", X_selected.shape[1])

# ----------------------------------------------------
# Calculate Mutual Information
# ----------------------------------------------------

mi_scores = mutual_info_classif(
    X_selected,
    y,
    discrete_features=True,
    random_state=42
)

# ----------------------------------------------------
# Create ranking
# ----------------------------------------------------

mi_df = pd.DataFrame({
    "Feature": X_selected.columns,
    "MI_Score": mi_scores
})

mi_df = mi_df.sort_values(
    by="MI_Score",
    ascending=False
).reset_index(drop=True)

print("\nTop 30 Most Informative APIs")
display(mi_df.head(30))

print("\nBottom 30 APIs")
display(mi_df.tail(30))

print("\nSummary")
print(mi_df["MI_Score"].describe())